# 🗂️ Lab: Relevance Scoring and Rerankers

## What this notebook does

This notebook builds a **Retrieval-Augmented Generation (RAG)** system that can answer questions by searching two sources:
1. **A PDF document** — the EU AI Act's "Living Repository of AI Literacy Practices" (real company case studies).
2. **A podcast transcript** — "The Blueprint for Trustworthy AI" (an audio episode auto-transcribed).

### The core problem RAG solves
Instead of relying on the AI's training knowledge, we:
1. Store our documents as **numerical vectors** (embeddings) in a database.
2. When a user asks a question, find the most relevant chunks by comparing vectors.
3. Pass those chunks to the LLM as context so it can answer accurately.

### What makes this notebook advanced
Basic RAG just uses **cosine similarity** to find relevant chunks. This notebook goes further:
- **LLM-based relevance scoring** — ask GPT to rate how relevant each chunk is (1-10).
- **Cross-encoder reranking** — use a dedicated ML model to re-score and reorder results.
- **Metadata filtering** — only search within specific document types or company categories.

---


## 📦 Step 1a — Imports & Environment Setup

Before we do anything, we need to import all the Python libraries we'll use throughout the notebook and load our secret API key.

**Key libraries:**
- `pypdf` — reads PDF files page by page
- `langchain` — the main framework for building RAG pipelines (splitting, embedding, retrieval)
- `openai` via `langchain_openai` — connects to GPT models and the embedding API
- `chromadb` via `langchain_community` — our local vector database
- `dotenv` — loads your `OPENAI_API_KEY` from a `.env` file so it stays secret


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# All library imports for the entire notebook
# ─────────────────────────────────────────────────────────────────────
import os       # file system operations (check if folders exist)
import re       # regular expressions for text pattern matching

# Loads variables from a .env file into the environment
# This keeps your API key out of the notebook code
from dotenv import load_dotenv

# Reads PDFs page by page and extracts plain text
from pypdf import PdfReader

# Splits long text into smaller overlapping chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The standard object LangChain uses to represent a text chunk + its metadata
from langchain_core.documents import Document

# Wraps OpenAI's embedding model (converts text → vectors)
# and GPT chat model (generates answers)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Our local vector database — stores and searches embeddings
from langchain_community.vectorstores import Chroma

# LangChain building blocks for constructing the RAG chain
from langchain_core.prompts import ChatPromptTemplate        # structured prompt templates
from langchain_core.runnables import RunnablePassthrough     # passes input unchanged through the chain
from langchain_core.output_parsers import StrOutputParser    # converts LLM output to a plain string

# For pretty-printing comparison tables
import pandas as pd

# Note: CrossEncoder is imported later in Step 4 to avoid slow startup

print("All imports successful.")


## Notebook logging setup

In [ ]:
import io, sys, contextlib
from datetime import datetime

LOG_FILE = "notebooklog.txt"

def _init_log():
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write("NOTEBOOK RUN LOG\n")
        f.write(f"Notebook : relevance_scoring_rerankersV2_with_logging.ipynb\n")
        f.write(f"Started  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 80 + "\n\n")

def log(label: str, output: str = "", file_created=None):
    """Append one entry to notebooklog.txt."""
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write("-" * 80 + "\n")
        f.write(f"[{datetime.now().strftime('%H:%M:%S')}]  {label}\n")
        f.write("-" * 80 + "\n")
        if file_created:
            for fname in ([file_created] if isinstance(file_created, str) else file_created):
                f.write(f"  FILE CREATED : {fname}\n")
            if output.strip():
                for line in output.strip().splitlines():
                    f.write(f"  {line}\n")
        elif output.strip():
            for line in output.strip().splitlines():
                f.write(f"  {line}\n")
        else:
            f.write("  (no output)\n")
        f.write("\n")

@contextlib.contextmanager
def capture():
    """Capture stdout so the real cell output can be both displayed AND logged."""
    buf = io.StringIO()
    old = sys.stdout
    sys.stdout = buf
    try:
        yield buf
    finally:
        sys.stdout = old
        print(buf.getvalue(), end="")

_init_log()
print(f"Logger ready — writing to '{LOG_FILE}'")

## Load the .env file

In [ ]:
# Load the .env file so OPENAI_API_KEY becomes available as an environment variable
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"

with capture() as out:
    print(f"OpenAI key loaded: {OPENAI_API_KEY[:8]}...")
log("Step 1a — API key loaded", out.getvalue())


## 📄 Step 1b — Load & Preview the PDF

We load the EU AI Act's Living Repository PDF using `pypdf`. Each page is extracted as plain text and stored in a list called `pdf_pages`.

**Why we join pages with `\n\n`**: This preserves the boundary between pages so we can later split the document intelligently.


In [ ]:
# Path to the PDF file (must be in the same folder as this notebook)
PDF_PATH = "Living_Repository_AI_Literacy_Practices.pdf"

# PdfReader opens the PDF and gives us access to each page
reader = PdfReader(PDF_PATH)
pdf_pages = []  # will hold the text of each page as a string

for page in reader.pages:
    text = page.extract_text() or ""  # extract_text() returns None for image-only pages
    # Re-encode as UTF-8 to strip characters that crash Windows terminals
    text = text.encode("utf-8", errors="ignore").decode("utf-8")
    pdf_pages.append(text)

# Join all pages into one big string, separated by double newlines
# This preserves page boundaries while keeping the text easy to process
pdf_text = "\n\n".join(pdf_pages)

with capture() as out:
    print(f"PDF loaded successfully.")
    print(f"Total pages      : {len(pdf_pages)}")
    print(f"Total characters : {len(pdf_text):,}")
    print(f"Total words      : {len(pdf_text.split()):,}")
    print(f"\n--- Preview: Page 1 ---")
    print(pdf_pages[0][:300])
log("Step 1b — PDF loaded", out.getvalue())


## 🧹 Step 1c — Clean PDF Text

Raw PDFs are messy. Every page has repeated headers, table-of-contents dot lines (`......`), and standalone page numbers that would pollute our search results.

This cell defines a `clean_pdf_text()` function that strips all of that noise using regular expressions (`re`). We also skip the first 4 pages (table of contents) since they contain no useful content.

**Key regex tricks used:**
- `re.match(r'^[\s\.]{5,}$', line)` — matches lines made only of dots/spaces (TOC noise)
- `re.match(r'^\s*\d+\s*$', line)` — matches standalone page numbers
- `re.sub(r'\s*\.{4,}\s*', ' ', line)` — removes inline dot sequences like `Section ........ 5`


In [ ]:
def clean_pdf_text(text: str) -> str:
    """
    Clean a single PDF page:

    - Remove repeated page headers (precise matching)
    - Remove lines that are purely dots/whitespace
    - Remove inline dot sequences (TOC artifacts)
    - Remove standalone page numbers
    """

    lines = text.split('\n')
    cleaned = []

    for line in lines:

        line = line.strip()

        # -------------------------------------------------
        # REMOVE REPEATED PAGE HEADER (PRECISE MATCH)
        # -------------------------------------------------

        # Header line 1
        if line == "Living Repository of":
            continue

        # Header line 2 (versioned title)
        if re.match(r"^AI Literacy Practices\s*–\s*v\.\s*\d{2}\.\d{2}\.\d{4}$", line):
            continue

        # -------------------------------------------------
        # REMOVE DOT-ONLY LINES (TOC noise)
        # -------------------------------------------------
        if re.match(r'^[\s\.]{5,}$', line):
            continue

        # -------------------------------------------------
        # REMOVE STANDALONE PAGE NUMBERS
        # -------------------------------------------------
        if re.match(r'^\s*\d+\s*$', line):
            continue

        # -------------------------------------------------
        # REMOVE INLINE DOT SEQUENCES
        # Example: "Section ........ 5"
        # -------------------------------------------------
        line = re.sub(r'\s*\.{4,}\s*', ' ', line).strip()

        # Skip empty lines after cleaning
        if not line or re.match(r'^\d+$', line):
            continue

        cleaned.append(line)

    return '\n'.join(cleaned)


# =====================================================
# APPLY CLEANING TO CONTENT PAGES
# =====================================================

SKIP_PAGES = 4
content_pages = pdf_pages[SKIP_PAGES:]

cleaned_pages = [clean_pdf_text(page) for page in content_pages]

full_cleaned_text = "\n\n".join(cleaned_pages)

with capture() as out:
    print(f"Pages used       : {len(content_pages)} (skipped first {SKIP_PAGES} TOC pages)")
    print(f"Cleaned length   : {len(full_cleaned_text):,} characters")
    print(f"\n--- First content page after cleaning ---")
    print(content_pages[0][:400])
log("Step 1c — PDF cleaned", out.getvalue())


## 🏢 Step 1d — Split PDF by Company & Extract Metadata

The PDF contains 28 company case studies, each starting with a recognisable pattern:
```
[Company Name]
On the organisation
Name: ...
```

This cell:
1. **`split_companies()`** — scans line by line to find these patterns and slices the text into per-company blocks.
2. **`extract_metadata()`** — uses regex to pull out structured fields (company name, size, sector, implementation status) from each block.
3. **`build_company_objects()`** — combines both functions into a clean list of `{company_text, metadata}` dictionaries.
4. Saves the segmented output to `company_segmentation.txt` for visual inspection.

> 💡 **Why metadata matters**: Later, we can use these fields to filter search results — e.g., "only show companies with `status: Fully implemented`".


In [ ]:
# =====================================================
# COMPANY SEGMENTATION + METADATA EXTRACTION
# Uses: full_cleaned_text (from CELL 4)
# =====================================================

import re


# -----------------------------------------------------
# STEP 1 — Split document into company blocks
# (same anti-slippage logic you already validated)
# -----------------------------------------------------

def split_companies(text):
    """
    Split full document into company-level blocks.

    Detects pattern:
        [Company Title]
        On the organisation
        Name:
    """

    text = text.replace("\r\n", "\n")
    lines = text.split("\n")

    company_starts = []

    for i in range(1, len(lines) - 2):

        current_line = lines[i].strip()
        next_line = lines[i + 1].strip()
        next_next_line = lines[i + 2].strip()

        if (
            next_line.lower() == "on the organisation"
            and next_next_line.lower().startswith("name:")
            and len(current_line) > 0
            and len(current_line.split()) <= 6
        ):
            company_starts.append(i)

    # End-of-document boundary
    company_starts.append(len(lines))

    companies = []

    for idx in range(len(company_starts) - 1):
        start = company_starts[idx]
        end = company_starts[idx + 1]

        company_text = "\n".join(lines[start:end]).strip()
        companies.append(company_text)

    return companies


# -----------------------------------------------------
# STEP 2 — Metadata extraction
# -----------------------------------------------------

def extract_metadata(company_text):
    """
    Extract company metadata from one company block.
    """

    def find(pattern):
        match = re.search(pattern, company_text, re.IGNORECASE)
        return match.group(1).strip() if match else "Unknown"

    metadata = {
        "doc_type": "pdf",
        "company_name": find(r"Name:\s*(.+)"),
        "size": find(r"Size:\s*(.+)"),
        "headquarter": find(r"Headquarter:\s*(.+)"),
        "sector": find(r"Sector:\s*(.+)"),
        "status": find(r"Status:\s*(.+)")
    }

    return metadata


# -----------------------------------------------------
# STEP 3 — Build structured company objects
# -----------------------------------------------------

def build_company_objects(text):
    """
    Output structure:

    [
      {
        "company_text": "...",
        "metadata": {...}
      }
    ]
    """

    company_objects = []

    companies = split_companies(text)

    for company_text in companies:

        metadata = extract_metadata(company_text)

        company_objects.append({
            "company_text": company_text,
            "metadata": metadata
        })

    return company_objects


# -----------------------------------------------------
# STEP 4 — RUN
# -----------------------------------------------------

company_objects = build_company_objects(full_cleaned_text)




# -----------------------------------------------------
# STEP 5 — Save company segmentation for inspection
# -----------------------------------------------------

def save_company_segmentation(company_objects, filename="company_segmentation.txt"):
    """
    Save company blocks + metadata to a text file
    for visual inspection.
    """

    with open(filename, "w", encoding="utf-8") as f:

        for i, obj in enumerate(company_objects, start=1):

            metadata = obj["metadata"]
            company_text = obj["company_text"]

            f.write("\n" + "="*70 + "\n")
            f.write(f"COMPANY BLOCK {i}\n")
            f.write("="*70 + "\n\n")

            f.write("METADATA\n")
            f.write("-"*20 + "\n")
            for k, v in metadata.items():
                f.write(f"{k}: {v}\n")

            f.write("\nFULL COMPANY TEXT\n")
            f.write("-"*20 + "\n")
            f.write(company_text)
            f.write("\n\n")


# Run save
save_company_segmentation(company_objects)

with capture() as out:
    print(f"Total companies detected: {len(company_objects)}")
    print(f"\n--- Example metadata ---")
    print(company_objects[0]["metadata"])
log("Step 1d — Company segmentation",
    out.getvalue(), file_created="company_segmentation.txt")


## ✂️ Step 1e — Chunk Companies into LangChain Documents

Each company block is split into **two chunks**:
- **Chunk 1** (`section: "organisation"`) — company background (size, sector, HQ, AI systems deployed).
- **Chunk 2** (`section: "ai_literacy"`) — the actual AI literacy practice description.

This two-chunk strategy means if a user asks "what AI literacy training does IBM do?", we retrieve the literacy chunk specifically — not a mix of background info.

Each chunk is wrapped in a **LangChain `Document`** object, which bundles:
- `page_content` — the text that will be embedded
- `metadata` — all the structured fields (company name, status, sector, etc.)

Result: **56 LangChain Documents** from 28 companies × 2 chunks each.


In [ ]:
# =====================================================
# TWO-CHUNK STRATEGY + LANGCHAIN DOCUMENTS
# Uses: company_objects (from CELL 5)
# =====================================================

from langchain_core.documents import Document
import re


# -----------------------------------------------------
# STEP 1 — Split ONE company into 2 chunks
# -----------------------------------------------------

def split_into_two_chunks(company_text):
    """
    Split company text into:

    Chunk 1 → On the organisation
    Chunk 2 → On the AI literacy approach (+ everything after)
    """

    # Find where AI literacy section starts
    match = re.search(r"\nOn the AI literacy approach", company_text)

    if not match:
        # fallback: whole text as one chunk
        return company_text.strip(), ""

    split_index = match.start()

    organisation_chunk = company_text[:split_index].strip()
    literacy_chunk = company_text[split_index:].strip()

    return organisation_chunk, literacy_chunk


# -----------------------------------------------------
# STEP 2 — Build LangChain Documents
# -----------------------------------------------------

def build_documents(company_objects):
    """
    Create LangChain Documents with:

    - page_content (chunk text)
    - metadata (company metadata)
    """

    documents = []

    for obj in company_objects:

        metadata = obj["metadata"]
        company_text = obj["company_text"]

        org_chunk, literacy_chunk = split_into_two_chunks(company_text)

        # ---- Chunk 1: Organisation ----
        if org_chunk:
            doc1 = Document(
                page_content=f"Company: {metadata['company_name']}\n{org_chunk}",
                metadata={
                    **metadata,
                    "section": "organisation"
                }
            )
            documents.append(doc1)

        # ---- Chunk 2: AI Literacy (full narrative) ----
        if literacy_chunk:
            doc2 = Document(
                page_content=f"Company: {metadata['company_name']}\n{literacy_chunk}",
                metadata={
                    **metadata,
                    "section": "ai_literacy"
                }
            )
            documents.append(doc2)

    return documents


# -----------------------------------------------------
# STEP 3 — RUN
# -----------------------------------------------------

documents = build_documents(company_objects)





# -----------------------------------------------------
# STEP 4 — Save chunking result for visual inspection
# -----------------------------------------------------

def save_company_chunking(documents, filename="company_chunking.txt"):
    """
    Save final chunks + metadata to file
    for visual inspection.
    """

    with open(filename, "w", encoding="utf-8") as f:

        for i, doc in enumerate(documents, start=1):

            f.write("\n" + "="*70 + "\n")
            f.write(f"CHUNK {i}\n")
            f.write("="*70 + "\n\n")

            f.write("METADATA\n")
            f.write("-"*20 + "\n")

            for k, v in doc.metadata.items():
                f.write(f"{k}: {v}\n")

            f.write("\nPAGE CONTENT\n")
            f.write("-"*20 + "\n")
            f.write(doc.page_content)
            f.write("\n\n")


# Run save
save_company_chunking(documents)

with capture() as out:
    print(f"Total documents created: {len(documents)}")
    print(f"\n--- Example Document ---")
    print(documents[0].page_content[:300])
log("Step 1e — PDF chunked into LangChain Documents",
    out.getvalue(), file_created="company_chunking.txt")


## 🎙️ Step 1f — Transcribe the Podcast

The podcast audio file (`.m4a`) is too large for the OpenAI Whisper API's 25MB limit, so we:
1. Use **FFmpeg** to split it into 10-minute chunks.
2. Send each chunk to **OpenAI Whisper** (`whisper-1`) which returns timestamped segments.
3. Reassemble into a single transcript file with corrected global timestamps.
4. Delete the temporary audio chunks.

Each transcript line looks like: `[00:01:45] This is what the speaker said.`

> 💡 **Note**: You only need to run this cell once. After the transcript is saved, the next cell reads from the `.txt` file directly.


In [ ]:
# =====================================================
# PODCAST TRANSCRIPTION (FFMPEG SPLIT — FIXED VERSION)
# =====================================================

from openai import OpenAI
import subprocess
import glob
import os

client = OpenAI(api_key=OPENAI_API_KEY)

AUDIO_PATH = "The_Blueprint_For_Trustworthy_AI.m4a"
FFMPEG_PATH = r"C:\ffmpeg\fullbuild\bin\ffmpeg.exe"

# -----------------------------------------------------
# STEP 1 — Split audio using FFmpeg
# -----------------------------------------------------

print("Splitting audio with ffmpeg...")

# creates files: chunk_000.m4a, chunk_001.m4a, ...
subprocess.run([
    FFMPEG_PATH,
    "-i", AUDIO_PATH,
    "-f", "segment",
    "-segment_time", "600",   # 10 minutes
    "-c", "copy",
    "chunk_%03d.m4a"
], check=True)

chunk_files = sorted(glob.glob("chunk_*.m4a"))

print(f"Created {len(chunk_files)} chunks")

# -----------------------------------------------------
# STEP 2 — Transcribe chunks
# -----------------------------------------------------

all_lines = []
global_offset = 0

for idx, chunk_file in enumerate(chunk_files):

    print(f"Transcribing {chunk_file}...")

    with open(chunk_file, "rb") as audio_file:

        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json"
        )

    # FIX: new SDK returns objects, not dicts
    for seg in transcript.segments:

        start = seg.start + global_offset
        text = seg.text.strip()

        h = int(start // 3600)
        m = int((start % 3600) // 60)
        s = int(start % 60)

        timestamp = f"[{h:02d}:{m:02d}:{s:02d}]"

        all_lines.append(f"{timestamp} {text}")

    # each chunk = 600 sec
    global_offset += 600


# -----------------------------------------------------
# STEP 3 — Save transcript
# -----------------------------------------------------

output_file = "podcast_transcript.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(all_lines))

print(f"\nTranscript saved to: {output_file}")

# -----------------------------------------------------
# STEP 4 — Cleanup temporary chunks
# -----------------------------------------------------

for f in chunk_files:
    os.remove(f)

print("Temporary chunks removed.")

print("\n--- Preview ---")
for line in all_lines[:10]:
    print(line)

with capture() as out:
    print(f"Transcription complete. Lines: {len(all_lines)}")
log("Step 1f — Podcast transcribed",
    out.getvalue(), file_created="podcast_transcript.txt")


## ✂️ Step 1g — Chunk Podcast Transcript into LangChain Documents

We group the transcript into chunks of **5 lines each** (about 15–30 seconds of speech).

Each chunk becomes a LangChain `Document` with:
- `page_content` — the 5 timestamped lines
- `metadata` — `doc_type: "podcast"`, source name, start/end timestamps

Result: **79 podcast Documents**.


In [ ]:
# =====================================================
# PODCAST CHUNKING (LEVEL-2 STRUCTURE)
# Uses: podcast_transcript.txt
# =====================================================

from langchain_core.documents import Document
import re

TRANSCRIPT_FILE = "podcast_transcript.txt"


# -----------------------------------------------------
# STEP 1 — Load transcript lines
# -----------------------------------------------------

with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]



# -----------------------------------------------------
# STEP 2 — Chunking strategy
# -----------------------------------------------------
# Group N timestamp lines per chunk
LINES_PER_CHUNK = 5


def extract_timestamp(line):
    """
    Extract [hh:mm:ss] from a line.
    """
    match = re.match(r"\[(\d{2}:\d{2}:\d{2})\]", line)
    return match.group(1) if match else "00:00:00"


# -----------------------------------------------------
# STEP 3 — Build Documents
# -----------------------------------------------------

podcast_documents = []

for i in range(0, len(lines), LINES_PER_CHUNK):

    chunk_lines = lines[i:i + LINES_PER_CHUNK]

    if not chunk_lines:
        continue

    start_time = extract_timestamp(chunk_lines[0])
    end_time = extract_timestamp(chunk_lines[-1])

    chunk_text = "\n".join(chunk_lines)

    doc = Document(
        page_content=chunk_text,
        metadata={
            "doc_type": "podcast",
            "source": "The_Blueprint_For_Trustworthy_AI",
            "start_time": start_time,
            "end_time": end_time
        }
    )

    podcast_documents.append(doc)



# -----------------------------------------------------
# STEP 4 — Save for visual inspection
# -----------------------------------------------------

def save_podcast_chunking(documents, filename="podcast_chunking.txt"):

    with open(filename, "w", encoding="utf-8") as f:

        for i, doc in enumerate(documents, start=1):

            f.write("\n" + "="*70 + "\n")
            f.write(f"PODCAST CHUNK {i}\n")
            f.write("="*70 + "\n\n")

            f.write("METADATA\n")
            f.write("-"*20 + "\n")
            for k, v in doc.metadata.items():
                f.write(f"{k}: {v}\n")

            f.write("\nPAGE CONTENT\n")
            f.write("-"*20 + "\n")
            f.write(doc.page_content)
            f.write("\n\n")


save_podcast_chunking(podcast_documents)


# Preview

with capture() as out:
    print(f"Total podcast chunks created: {len(podcast_documents)}")
    print(f"\n--- Preview first chunk ---")
    print(podcast_documents[0].page_content[:200])
log("Step 1g — Podcast chunked into LangChain Documents",
    out.getvalue(), file_created=["podcast_transcript.txt", "podcast_chunking.txt"])


## 🔗 Step 2a — Combine All Documents

We merge the 56 PDF documents and 79 podcast documents into one list: `all_docs` (135 total).

This cell also does a **sanity check** — printing the source breakdown and previewing one document from each type to make sure the metadata is correct before we do the expensive embedding step.


In [ ]:
# =====================================================
# COMBINE ALL DOCUMENTS (PDF + PODCAST)
# =====================================================

# Combine all documents into one list
all_docs = documents + podcast_documents

with capture() as out:
    print(f"Total chunks ready for embedding: {len(all_docs)}")
    print(f"  PDF chunks     : {len(documents)}")
    print(f"  Podcast chunks : {len(podcast_documents)}")
    print(f"\n--- Metadata breakdown (doc_type) ---")
    sources = {}
    for doc in all_docs:
        k = doc.metadata.get("doc_type", "unknown")
        sources[k] = sources.get(k, 0) + 1
    for k, v in sources.items():
        print(f"  {k}: {v} chunks")
    print(f"\n--- PDF chunk example ---")
    pdf_ex = next(d for d in all_docs if d.metadata.get("doc_type") == "pdf")
    print(f"  Company : {pdf_ex.metadata.get('company_name')}")
    print(f"  Section : {pdf_ex.metadata.get('section')}")
    print(f"  Content : {pdf_ex.page_content[:150].replace(chr(10), ' ')}")
    print(f"\n--- Podcast chunk example ---")
    pod_ex = next(d for d in all_docs if d.metadata.get("doc_type") == "podcast")
    print(f"  Start   : {pod_ex.metadata.get('start_time')}")
    print(f"  Content : {pod_ex.page_content[:150].replace(chr(10), ' ')}")
log("Step 2a — All documents combined", out.getvalue())

## 🔢 Step 2b — Create the Embedding Model

An **embedding model** converts text into a list of numbers (a vector) that captures the semantic meaning of the text. Similar texts will have similar vectors.

We use OpenAI's `text-embedding-3-small` — a fast, cheap, and high-quality embedding model.

> 💡 Think of each document being converted into a point in a 1536-dimensional space. When you search, your query is also converted to a point, and we find the closest document points.


In [ ]:
# Initialize the OpenAI embedding model
# 'text-embedding-3-small' produces 1536-dimensional vectors
# It's cheaper and faster than 'text-embedding-3-large' while still being high quality
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

with capture() as out:
    print("Embedding model ready: text-embedding-3-small")
log("Step 2b — Embedding model initialized", out.getvalue())


## 🗄️ Step 2c — Build the Vector Store (ChromaDB)

We store all 135 document embeddings in **ChromaDB**, a local vector database saved to the `chroma_db/` folder.

**Smart loading logic:** If the database already exists on disk, we load it (saves API costs and time). If not, we build it from scratch by calling the OpenAI embedding API for each document.

After this step, we do a quick test query to verify everything works — we ask "What are the key principles of trustworthy AI?" and check that we get back sensible results.


In [ ]:
# Path where ChromaDB will save/load the vector database on disk
CHROMA_PATH = "chroma_db"

# Check if we already have a saved database to avoid re-embedding everything
if os.path.exists(CHROMA_PATH) and os.listdir(CHROMA_PATH):

    # Load the existing database from disk (no API calls needed)
    vectorstore = Chroma(
        persist_directory=CHROMA_PATH,          # folder containing the saved DB
        embedding_function=embedding_model       # needed to embed future queries
    )


else:
    # First run — call OpenAI API to embed all 135 documents, then save to disk

    vectorstore = Chroma.from_documents(
        documents=all_docs,            # list of LangChain Document objects
        embedding=embedding_model,     # model that converts text → vectors
        persist_directory=CHROMA_PATH  # save to disk so we don't re-embed next time
    )

with capture() as out:
    print(f"Vector store ready. Total vectors: {vectorstore._collection.count()}")
log("Step 2c — Vector store built/loaded", out.getvalue())


In [ ]:
# Quick test: search for the top 4 most similar chunks to this query
with capture() as out:
    test_results = vectorstore.similarity_search_with_score(
        "What are the key principles of trustworthy AI?", k=4)
    for doc, score in test_results:
        print(f"Score: {score:.4f} | {doc.metadata.get('doc_type','?')} | {doc.page_content[:80].replace(chr(10),' ')}...")
log("Step 2c — Baseline similarity search test", out.getvalue())


## 🤖 Step 3 — LLM-Based Relevance Scoring (Advanced)

### The problem with basic similarity search
Vector similarity (cosine distance) measures how *similar* text looks statistically. But "similar" doesn't always mean "relevant" to your question.

For example: a chunk that says "We trust AI systems" has high semantic similarity to "trustworthy AI principles" but provides no useful information about what those principles actually *are*.

### The solution: ask the LLM
We send each retrieved chunk to GPT with the prompt:
> "Rate from 1 to 10 how relevant this text is to the query. Return ONLY the number."

This gives us a **relevance score** that measures genuine usefulness — not just statistical similarity.

### How the scores are combined
Each chunk now has two scores:
- **`sim_score`** — the cosine distance from the vector store (lower = more similar)
- **`rel_score`** — the LLM's 1-10 relevance rating (higher = more relevant)

We then **sort by `rel_score`** to reorder the results, and also try a **combined score** (`rel_score - sim_score`) that balances both signals.

> ⚠️ **Cost trade-off**: This approach makes one extra LLM API call per retrieved chunk. For production, you'd only do this on the top-k results.


In [ ]:
def llm_relevance_score(query, text):
    """
    Ask the LLM to rate how relevant a text chunk is to a query.
    Returns a float between 1.0 and 10.0.
    """
    # Build a simple prompt that asks for just a number
    prompt = f"""
Rate from 1 to 10 how relevant this text is to the query.

Query:
{query}

Text:
{text}

Return ONLY the number.
"""
    # Send to the LLM and parse the response as a float
    response = llm.invoke(prompt)
    return float(response.content.strip())

with capture() as out:
    print("llm_relevance_score() defined — ready to use")
log("Step 3 — llm_relevance_score() defined", out.getvalue())


In [ ]:
#Step 3 — LLM relevance score (single chunk test)

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

results = vectorstore.similarity_search_with_score(
    "What are the key principles of trustworthy AI?", k=1)
doc, sim_score = results[0]

with capture() as out:
    score = llm_relevance_score("What are the key principles of trustworthy AI?", doc.page_content)
    print(f"Chunk  : {doc.page_content[:120].replace(chr(10),' ')}")
    print(f"Sim    : {sim_score:.4f}")
    print(f"LLM    : {score}")
log("Step 3 — LLM relevance score (single chunk test)", out.getvalue())


In [ ]:
# Retrieve top 4 similar chunks and score each with the LLM
results = vectorstore.similarity_search_with_score(
    "What are the key principles of trustworthy AI?", k=4)

scored_results = []
with capture() as out:
    print(f"{'Chunk (first 60 chars)':<62} {'sim':>6}  {'llm':>4}")
    print("-" * 76)
    for doc, sim_score in results:
        rel_score = llm_relevance_score(
            "What are the key principles of trustworthy AI?", doc.page_content)
        scored_results.append((doc, sim_score, rel_score))
        snippet = doc.page_content[:60].replace("\n", " ")
        print(f"{snippet:<62} {sim_score:>6.4f}  {rel_score:>4.1f}")
log("Step 3 — All 4 chunks scored (sim + LLM relevance)", out.getvalue())


In [ ]:
# Sort by LLM relevance score (highest first)
reranked = sorted(scored_results, key=lambda x: x[2], reverse=True)

with capture() as out:
    print("Reranked by LLM relevance score:")
    print(f"{'#':<3} {'rel':>4}  {'sim':>6}  {'chunk (first 70 chars)'}")
    print("-" * 80)
    for rank, (doc, sim, rel) in enumerate(reranked, 1):
        snippet = doc.page_content[:70].replace("\n", " ")
        print(f"{rank:<3} {rel:>4.1f}  {sim:>6.4f}  {snippet}")
log("Step 3 — Reranked by LLM relevance score", out.getvalue())


In [ ]:
# Combined score: final = rel_score - sim_score
# Subtracting sim_score (a distance) rewards chunks that are both
# relevant AND close in vector space.
combined = []
for doc, sim_score, rel_score in scored_results:
    final_score = rel_score - sim_score
    combined.append((doc, sim_score, rel_score, final_score))

combined_sorted = sorted(combined, key=lambda x: x[3], reverse=True)

with capture() as out:
    print("Reranked by combined score (rel - sim):")
    print(f"{'#':<3} {'final':>6}  {'rel':>4}  {'sim':>6}  {'chunk (first 60 chars)'}")
    print("-" * 80)
    for rank, (doc, sim, rel, final) in enumerate(combined_sorted, 1):
        snippet = doc.page_content[:60].replace("\n", " ")
        print(f"{rank:<3} {final:>6.3f}  {rel:>4.1f}  {sim:>6.4f}  {snippet}")
log("Step 3 — Combined score reranking", out.getvalue())


## ⚡ Step 4 — Cross-Encoder Reranking (Advanced)

### What is a Cross-Encoder?

The LLM scoring in Step 3 works but is slow and expensive (one API call per chunk). A **Cross-Encoder** is a specialised ML model trained specifically to score query-document relevance — much faster and cheaper.

**How it works:**
- Input: a `(query, document)` pair
- Output: a single relevance score (can be negative — it's a raw logit, not a probability)

We use `cross-encoder/ms-marco-MiniLM-L-6-v2` — a small model trained on the MS MARCO passage retrieval dataset that is well-suited for question answering.

### What this section does
1. Loads the Cross-Encoder model from HuggingFace.
2. Creates `(query, document_text)` pairs for each retrieved chunk.
3. Runs `reranker.predict(pairs)` to get a score for each pair.
4. Sorts results by Cross-Encoder score.
5. **Compares** the original similarity order vs. the reranked order.

> 💡 Notice how the order changes after reranking! The Cross-Encoder promotes chunks that directly answer the question, even if they weren't the most similar vectors.


In [ ]:
# Import the CrossEncoder class from the sentence-transformers library
# (This import is done here rather than at the top to avoid slow startup)
from sentence_transformers import CrossEncoder

# Download and load the reranker model from HuggingFace
# 'ms-marco-MiniLM-L-6-v2' is trained on Microsoft's MARCO passage retrieval dataset
# It's small (~90MB), fast, and excellent for query-document relevance scoring
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

with capture() as out:
    print("CrossEncoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")
log("Step 4 — CrossEncoder loaded", out.getvalue())


In [ ]:
# CrossEncoder raw scores
query = "What are the key principles of trustworthy AI?"

# Build (query, document) pairs for the CrossEncoder
pairs = [(query, doc.page_content) for doc, _, _ in scored_results]

# Score all pairs in one batch — returns raw logits (can be negative)
rerank_scores = reranker.predict(pairs)

with capture() as out:
    print("CrossEncoder raw scores (logits):")
    for (doc, sim, rel), score in zip(scored_results, rerank_scores):
        snippet = doc.page_content[:65].replace("\n", " ")
        print(f"  {score:+.4f}  {snippet}")
log("Step 4 — CrossEncoder raw scores", out.getvalue())


In [ ]:
# Attach CrossEncoder scores and sort highest-first
reranked = [(doc, sim, rel, float(rr))
            for (doc, sim, rel), rr in zip(scored_results, rerank_scores)]
reranked_sorted = sorted(reranked, key=lambda x: x[3], reverse=True)

with capture() as out:
    print("Reranked by CrossEncoder score:")
    print(f"{'#':<3} {'ce':>7}  {'llm':>4}  {'sim':>6}  {'chunk (first 60 chars)'}")
    print("-" * 80)
    for rank, (doc, sim, rel, rr) in enumerate(reranked_sorted, 1):
        snippet = doc.page_content[:60].replace("\n", " ")
        print(f"{rank:<3} {rr:>+7.4f}  {rel:>4.1f}  {sim:>6.4f}  {snippet}")
log("Step 4 — Results reranked by CrossEncoder", out.getvalue())


In [ ]:
# Side-by-side: original vector similarity order vs cross-encoder order
with capture() as out:
    print("BASELINE ORDER (vector similarity):")
    for rank, (doc, sim, rel) in enumerate(scored_results, 1):
        print(f"  #{rank}  sim={sim:.4f}  {doc.page_content[:70].replace(chr(10),' ')}")
    print()
    print("RERANKED ORDER (cross-encoder):")
    for rank, (doc, sim, rel, rr) in enumerate(reranked_sorted, 1):
        print(f"  #{rank}  ce={rr:+.4f}  {doc.page_content[:70].replace(chr(10),' ')}")
log("Step 4 — Baseline vs CrossEncoder order comparison", out.getvalue())


In [ ]:
# Compare all three ranking methods
print("LLM relevance order (score = 1-10 rating):")
for rank, (doc, sim, rel, rr) in enumerate(reranked, 1):
    print(rank, rel, doc.page_content[:60])

print("\nCross-encoder rerank order (score = raw logit):")
for rank, (doc, sim, rel, rr) in enumerate(reranked_sorted, 1):
    print(rank, rr, doc.page_content[:60])

with capture() as out:
    print("LLM relevance order (score = 1-10 rating):")
    for rank, (doc, sim, rel, rr) in enumerate(reranked, 1):
        print(f"  #{rank}  llm={rel:>4.1f}  {doc.page_content[:65].replace(chr(10),' ')}")
    print()
    print("Cross-encoder rerank order (score = raw logit):")
    for rank, (doc, sim, rel, rr) in enumerate(reranked_sorted, 1):
        print(f"  #{rank}  ce={rr:+.4f}  {doc.page_content[:65].replace(chr(10),' ')}")
log("Step 4 — LLM vs CrossEncoder method comparison", out.getvalue())


## 🔍 Step 5 — Metadata Filtering

### Why filtering matters

Without filtering, a search for "which companies fully implemented AI literacy?" might return:
- Companies with `status: "Partially rolled-out"` (wrong)
- Podcast chunks (irrelevant)

**Metadata filtering** lets us restrict the search to only documents matching specific criteria — like `status = "Fully implemented"` or `doc_type = "pdf"`.

### What this section does
1. Demonstrates a **raw filtered search** using ChromaDB's `filter=` parameter.
2. Shows **without filter** (baseline) — returns a mix of sources including wrong statuses.
3. Defines `run_filtered_search()` — a helper that accepts any metadata filter dictionary.
4. Defines `smart_search()` — an intelligent wrapper that detects query intent and applies the right filter automatically (e.g., if the query mentions "fully implemented", apply that filter).
5. Runs all 4 test queries through `smart_search()` to compare results.


In [ ]:
# Demonstration: search without any filter (baseline)
# Notice it returns companies with different implementation statuses mixed together
with capture() as out:
    r = vectorstore.similarity_search_with_score(
        "Which organisations have fully implemented AI literacy practices?", k=4)
    print("No-filter baseline — notice mixed statuses:")
    for doc, score in r:
        company = doc.metadata.get("company_name", "podcast")
        status  = doc.metadata.get("status", "—")
        print(f"  {score:.4f}  {company:<38}  status={status}")
log("Step 5 — Baseline search without filter", out.getvalue())


In [ ]:
# --- Test queries ---
TEST_QUERIES = [
    "What are the key principles of trustworthy AI?",
    "What are small companies doing for AI literacy?",
    "How is AI transparency being implemented in practice?",
    "Which organisations have fully implemented AI literacy practices?"
]

# --- Baseline similarity search ---
def run_baseline_search(query: str, k: int = 50) -> list:
    """Run similarity search and return results with scores."""
    return vectorstore.similarity_search_with_score(query, k=k)

# --- Display helper ---
def display_results(query: str, results: list):
    """Print search results in a readable format."""
    print(f"\nQuery: {query}")
    for i, (doc, score) in enumerate(results, 1):
        source = doc.metadata.get("doc_type", "unknown")
        company = doc.metadata.get("company_name", "")
        label = f"{source} — {company}" if company else source
        print(f"  [{i}] Score: {score:.4f} | {label}")
        print(f"       {doc.page_content[:120].replace(chr(10), ' ')}...")

with capture() as out:
    print("Test queries defined:")
    for idx, q in enumerate(TEST_QUERIES, 1):
        print(f"  Q{idx}: {q}")
log("Step 5 — Test queries and helper functions defined", out.getvalue())

In [ ]:
#Filtered search

def run_filtered_search(query: str, k: int = 10, metadata_filter: dict = None):
    """
    Run a vector similarity search with an optional metadata filter.
    
    The filter is applied BEFORE similarity ranking — only documents
    matching all filter conditions are considered.
    
    Example filter: {"status": "Fully implemented"}
    """
    return vectorstore.similarity_search_with_score(
        query,
        k=k,
        filter=metadata_filter   # ChromaDB applies this as a WHERE clause
    )

# Test it: restrict results to companies with 'Fully implemented' status
results = run_filtered_search(
    "Which organisations have fully implemented AI literacy practices?",
    metadata_filter={"status": "Fully implemented"}
)

# Demo: restrict to fully-implemented companies
with capture() as out:
    r = run_filtered_search(
        "Which organisations have fully implemented AI literacy practices?",
        metadata_filter={"status": "Fully implemented"})
    print("Filtered (status=Fully implemented):")
    for doc, score in r:
        company = doc.metadata.get("company_name", "?")
        print(f"  {score:.4f}  {company}")
log("Step 5 — run_filtered_search() defined and tested", out.getvalue())


In [ ]:
# Smart search and filter

def smart_search(query: str, k: int = 10):
    """
    Intelligent search that automatically applies metadata filters
    based on keywords detected in the query.
    
    Currently handles:
    - 'fully implemented' → filters by status = 'Fully implemented'
    - Everything else → plain similarity search
    
    In a production system, you could use an LLM to classify the query
    and choose filters dynamically.
    """
    if "fully implemented" in query.lower():  # simple keyword detection
        return run_filtered_search(
            query,
            k=k,
            metadata_filter={"status": "Fully implemented"}  # apply the filter
        )
    return run_baseline_search(query, k=k)  # no filter needed

# Test smart_search with the filter-triggering query
results = smart_search(
    "Which organisations have fully implemented AI literacy practices?"
)

# Test: query that triggers the filter
with capture() as out:
    r = smart_search("Which organisations have fully implemented AI literacy practices?")
    print("smart_search — filter triggered (status=Fully implemented):")
    for doc, score in r:
        company = doc.metadata.get("company_name", "?")
        print(f"  {score:.4f}  {company}")
log("Step 5 — smart_search() defined and tested (filter triggered)", out.getvalue())


In [ ]:
# Test smart_search with a query that does NOT trigger any filter
with capture() as out:
    results = smart_search("What are the key principles of trustworthy AI?")
    print("smart_search — no filter triggered:")
    for doc, score in results:
        src_type = doc.metadata.get("doc_type", "")
        company  = doc.metadata.get("company_name", "")
        label    = f"{src_type} — {company}" if company else src_type
        print(f"  {score:.4f}  {label}  |  {doc.page_content[:60].replace(chr(10), ' ')}")
log("Step 5 — smart_search test (no filter)", out.getvalue())


In [ ]:
# All test queries thorugh smart_serach

smart_results = {}

with capture() as out:
    for query in TEST_QUERIES:
        results = smart_search(query)
        smart_results[query] = results
        print(f"\n{'='*70}")
        print(f"Query: {query}")
        print(f"{'='*70}")
        for doc, score in results:
            company  = doc.metadata.get("company_name", "")
            src_type = doc.metadata.get("doc_type", "")
            label    = f"{src_type} — {company}" if company else src_type
            preview  = doc.page_content[:60].replace(chr(10), " ")
            print(f"  {score:.4f}  {label}  |  {preview}")
log("Step 5 — All test queries through smart_search", out.getvalue())


## 🔧 Step 6 — Complete RAG Pipeline with Reranking

### Putting it all together

Now we wire everything into a full end-to-end RAG pipeline:

1. **`smart_retriever(query, k=50)`** — runs `smart_search()` (with metadata filtering) and returns the top 50 documents.
2. **`rerank_docs(query, docs)`** — takes those 50 docs, runs the Cross-Encoder on all of them, and returns the top 5 in relevance order.
3. **`ask_rag(query)`** — combines both: retrieves → reranks → formats as context → calls GPT to generate a final answer.

### Why retrieve 50 but only pass 5 to the LLM?
- Retrieving 50 gives us a wide net to catch all relevant content.
- But passing 50 full chunks to the LLM would be expensive and noisy.
- The reranker picks the best 5 before the LLM sees anything.

> 💡 This "retrieve broad, rerank narrow" pattern is the gold standard for production RAG systems.


In [ ]:
def rerank_docs(query: str, docs: list, top_k: int = 50) -> list:
    """
    Given a query and a list of documents, use the Cross-Encoder
    to score each (query, document) pair and return the top_k
    most relevant documents in ranked order.
    """
    # Build pairs: the Cross-Encoder needs both the query AND the document text together
    pairs = [(query, d.page_content) for d in docs]

    # Get relevance scores for all pairs in one batch
    scores = reranker.predict(pairs)

    # Combine each document with its score, then sort highest-first
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    # Return only the documents (discard scores), keeping top_k
    return [doc for doc, _ in ranked[:top_k]]

with capture() as out:
    print("rerank_docs() defined — ready to use")
log("Step 6 — rerank_docs() defined", out.getvalue())


In [ ]:
def smart_retriever(query: str, k: int = 50):
    """
    Step 1 of the pipeline: cast a wide net.
    Retrieve k=50 documents using smart_search (with metadata filtering).
    Returns just the Document objects (strips the scores).
    """
    results = smart_search(query, k=k)  # returns list of (doc, score) tuples
    return [doc for doc, _ in results]  # keep only the documents

with capture() as out:
    print("smart_retriever() defined — ready to use")
log("Step 6 — smart_retriever() defined", out.getvalue())


In [ ]:
# Test the retriever: how many docs come back?
docs = smart_retriever("Which organisations have fully implemented AI literacy practices?")

with capture() as out:
    print(f"Docs returned : {len(docs)}")
    print(f"First metadata: {docs[0].metadata}")
log("Step 6 — smart_retriever() tested", out.getvalue())


In [ ]:
# Initialize the LLM once here — used by both llm_relevance_score (Step 3)
# and ask_rag (Step 6). Defining it early avoids NameError later.
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

with capture() as out:
    print("LLM initialized: gpt-4o-mini")
log("Step 6 setup — LLM initialized", out.getvalue())


In [ ]:
def ask_rag(query: str):
    """
    Full RAG pipeline:
    1. Retrieve candidate documents using smart_retriever (with filters)
    2. Rerank with Cross-Encoder, keep top 5
    3. Concatenate the top 5 as context
    4. Ask GPT to answer using ONLY that context
    """
    # Step 1: Retrieve broad set of candidates
    docs = smart_retriever(query)

    # Step 2: Rerank to get the 5 best
    top_docs = rerank_docs(query, docs, top_k=5)

    # Step 3: Combine the document texts into one context string
    context = "\n\n".join([d.page_content for d in top_docs])

    # Step 4: Build the prompt
    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""
    # Step 5: Send to GPT and return the response
    return llm.invoke(prompt)

with capture() as out:
    print("ask_rag() defined — full pipeline: smart_retriever → rerank_docs → LLM")
log("Step 6 — ask_rag() defined (full pipeline)", out.getvalue())

## 📊 Step 7 — Evaluation: Baseline vs. Smart Search

### Comparing retrieval quality manually

We run the same 4 test queries through both:
- **Baseline search** — plain vector similarity, no filtering, no reranking
- **Smart search** — metadata filtering applied when relevant

Results are displayed with source labels so you can visually judge which approach returns more useful chunks.

### Key things to notice
- For "which organisations have fully implemented...?", the **baseline** returns partially-rolled-out companies. The **smart search** filters them out.
- For "what are the key principles of trustworthy AI?", both approaches return podcast chunks — which is correct since that's where the conceptual content lives.
- The `Source distribution` at the bottom of each result shows whether the system correctly mixed or separated sources.


In [ ]:
results = vectorstore.similarity_search_with_score(
    "Which organisations have fully implemented AI literacy practices?",
    k=20,
    filter={"status": "Fully implemented"}
)

with capture() as out:
    print("Filtered retrieval (status=Fully implemented, k=20):")
    for doc, score in results:
        print(f"  {score:.4f}  {doc.metadata.get('company_name','?')}")
log("Step 7 — Filtered retrieval evaluation", out.getvalue())


In [ ]:
companies = set()
for doc, _ in results:
    companies.add(doc.metadata["company_name"])

with capture() as out:
    print(f"Unique companies with 'Fully implemented' status ({len(companies)} total):")
    for c in sorted(companies):
        print(f"  {c}")
log("Step 7 — Unique companies retrieved", out.getvalue())


In [ ]:
def ask_baseline(query: str):
    """Simple RAG with no reranking — just top 5 from vector search directly."""
    results = vectorstore.similarity_search(query, k=5)
    context = "\n\n".join([d.page_content for d in results])
    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""
    return llm.invoke(prompt)

with capture() as out:
    print("ask_baseline() defined — plain vector search, no reranking")
log("Step 7 — ask_baseline() defined", out.getvalue())

## 💬 Step 7b — Final Q&A: Ask the Complete RAG System

These last cells test the full pipeline with two example questions and print the LLM's generated answers.

**Query 1**: "Which organisations have fully implemented AI literacy practices?"  
→ Expects a list of company names from the PDF document.

**Query 2**: "What are the key principles of trustworthy AI?"  
→ Expects a conceptual answer sourced from the podcast transcript.

These outputs let you **manually evaluate** answer quality — the gold standard when you don't have pre-labelled test data.


In [ ]:
with capture() as out:
    query = "Which organisations have fully implemented AI literacy practices?"
    print("=" * 70)
    print(f"Query: {query}")
    print("=" * 70)
    print("\n--- BASELINE (plain vector search, no reranking) ---")
    print(ask_baseline(query).content)
    print("\n--- WITH RERANKING (smart_retriever + Cross-Encoder) ---")
    print(ask_rag(query).content)
log("Step 7 — Comparison baseline vs reranked: fully implemented organisations",
    out.getvalue())

In [ ]:
with capture() as out:
    query = "What are the key principles of trustworthy AI?"
    print("=" * 70)
    print(f"Query: {query}")
    print("=" * 70)
    print("\n--- BASELINE (plain vector search, no reranking) ---")
    print(ask_baseline(query).content)
    print("\n--- WITH RERANKING (smart_retriever + Cross-Encoder) ---")
    print(ask_rag(query).content)
log("Step 7 — Comparison baseline vs reranked: key principles of trustworthy AI",
    out.getvalue())
